In [2]:
import numpy as np
import sys

def convert_safe(input_path, output_path, dimension=50):
    print(f"Processing {input_path}...")
    
    # We will buffer output to write properly
    with open(input_path, 'r', encoding='utf-8', errors='ignore') as f_in, \
         open(output_path, 'wb') as f_out:
        
        # 1. First, count lines for the header
        # (This might take a moment but it's necessary for the binary header)
        lines = f_in.readlines()
        vocab_size = len(lines)
        
        print(f"Detected Vocab: {vocab_size}, Dimensions: {dimension}")
        
        # 2. Write the Header
        # Format: "vocab_size dimension\n"
        f_out.write(f"{vocab_size} {dimension}\n".encode('utf-8'))
        
        # 3. Process each line safely
        skipped = 0
        for i, line in enumerate(lines):
            parts = line.strip().split()
            
            # Safety Check: We need at least word + 50 numbers
            if len(parts) <= dimension:
                skipped += 1
                continue
                
            try:
                # TRICK: Grab the LAST 50 items as the vector
                vector_str = parts[-dimension:]
                vector = np.array(vector_str, dtype=np.float32)
                
                # Everything BEFORE the last 50 items is the word
                # Join them just in case it was "New York" -> "New_York"
                word_raw = "_".join(parts[:-dimension])
                
                # Write Word + Space
                f_out.write(f"{word_raw} ".encode('utf-8'))
                
                # Write Vector
                f_out.write(vector.tobytes())
                
                # Write Newline
                f_out.write(b'\n')
                
            except ValueError:
                # If a line is truly broken, just skip it
                skipped += 1
                continue

    print(f"Done! Saved to {output_path}")
    print(f"Skipped {skipped} problematic lines.")

# --- RUN IT ---
# Make sure to match the filename exactly!
filename = 'wiki_giga_2024_50_MFT20_vectors_seed_123_alpha_0.75_eta_0.075_combined.txt'
convert_safe(filename, 'model.bin', dimension=50)

Processing wiki_giga_2024_50_MFT20_vectors_seed_123_alpha_0.75_eta_0.075_combined.txt...
Detected Vocab: 1291147, Dimensions: 50
Done! Saved to model.bin
Skipped 0 problematic lines.
